Function: synchronize IMU + odometry onto one timeline.

"""
Resample ACC, GYRO, ODO to a common sample rate using interpolation. (According to QC)

Input: Labeled runs from data/labeled/ (or raw runs from data/raw/)
Output: Resampled runs with uniform dt for all sensors

Strategy:
1. Determine target sample rate (e.g., 100 Hz based on your QC pass criteria)
2. Create unified time grid from min(t_rel) to max(t_rel)
3. Interpolate each sensor to the grid (linear or nearest-neighbor)
4. Preserve label column during resampling
5. Save to data/resampled/
"""


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from dataclasses import replace

# Make project root importable
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / 'src').exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.io import load_run, save_labeled_run, discover_run_dirs

In [2]:
# Load all labeled runs from data/labeled/
labeled_data_dir = PROJECT_ROOT / "data" / "labeled"
labeled_run_dirs = sorted(labeled_data_dir.glob("log_*"))

labeled_runs = {}
for run_dir in labeled_run_dirs:
    print(f"Loading {run_dir.name}...")
    try:
        run = load_run(run_dir, include_pose=False)  # Exclude pose
        labeled_runs[run.run_id] = run
        print(f"Loaded {run.run_id}: ACC={len(run.acc)}, GYRO={len(run.gyro)}, ODO={len(run.odo)}")
    except Exception as e:
        print(f"Failed to load {run_dir.name}: {e}")

print(f"\nTotal labeled runs: {len(labeled_runs)}")

Loading log_20260216_114652.530...


/Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/src/io.py:61: DtypeWarning: Columns (0: label) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Loaded log_20260216_114652.530: ACC=145708, GYRO=147427, ODO=219888
Loading log_20260223_142511.490...
Loaded log_20260223_142511.490: ACC=243359, GYRO=243378, ODO=365121

Total labeled runs: 2
